# Supervised Learning (Módulo 6) – Classificação de MalwareFamily

Treinam-se vários algoritmos de classificação com o **mesmo pré-processamento** do `data_preparation.ipynb` e comparam-se na **validação**. O conjunto de teste não é usado aqui (fica para a avaliação final, no Módulo 7).

Modelos: Logistic Regression, Decision Tree, Random Forest e XGBoost.
Métrica principal: **F1 macro** (a accuracy engana com classes tão desequilibradas).

## 1. Preparação (igual ao `data_preparation.ipynb`)

In [ ]:
%pip install -q kagglehub scikit-learn pandas numpy xgboost

import glob, os, time
import numpy as np
import pandas as pd
import kagglehub

TARGET, GROUP, SEED = "MalwareFamily", "Hash", 42

path = kagglehub.dataset_download(
    "raneemababneh/iot-malware-network-traffic-dataset-cic-ynu-2026"
)
df = pd.read_csv(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)[0])

# --- Data cleaning ---
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates().reset_index(drop=True)
constantes = [c for c in df.select_dtypes("number") if df[c].nunique(dropna=True) <= 1]
drop_cols = list(dict.fromkeys([TARGET, GROUP, "Arch", "Tot size", "LLC"] + constantes))

# --- Famílias raras (< MIN_HASHES hashes) juntadas em "Other_rare" ---
MIN_HASHES = 10
h = df.groupby(TARGET)[GROUP].nunique()
df[TARGET] = df[TARGET].replace({r: "Other_rare" for r in h[h < MIN_HASHES].index})

# --- Split 80/10/10, estratificado e por grupos (Hash) ---
from sklearn.model_selection import StratifiedGroupKFold
X, y, groups = df.drop(columns=drop_cols), df[TARGET], df[GROUP]

tr, tmp = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, groups))
X_train, y_train = X.iloc[tr], y.iloc[tr]
X_tmp, y_tmp, g_tmp = X.iloc[tmp], y.iloc[tmp], groups.iloc[tmp]
va, te = next(StratifiedGroupKFold(2, shuffle=True, random_state=SEED).split(X_tmp, y_tmp, g_tmp))
X_val, y_val = X_tmp.iloc[va], y_tmp.iloc[va]
X_test, y_test = X_tmp.iloc[te], y_tmp.iloc[te]   # NÃO usar neste notebook

print("Treino:", len(X_train), "| Validação:", len(X_val), "| Teste:", len(X_test))

## 2. Amostra de treino

Treinar 4 modelos em ~590 mil linhas é lento. Para a **comparação** usa-se uma amostra estratificada do treino (as famílias raras mantêm-se representadas). A validação é sempre completa. Depois de escolhido o melhor modelo, pode treinar-se com o treino todo (`TRAIN_SAMPLE = None`).

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_SAMPLE = 150000   # None = usar o treino todo

if TRAIN_SAMPLE:
    X_fit, _, y_fit, _ = train_test_split(
        X_train, y_train, train_size=TRAIN_SAMPLE, stratify=y_train, random_state=SEED)
else:
    X_fit, y_fit = X_train, y_train

print("Linhas usadas no treino:", len(X_fit))
print(y_fit.value_counts())

## 3. Pré-processamento e modelos

O pré-processamento é o do Módulo 4 (`log1p` nas colunas enviesadas → imputação pela mediana → `StandardScaler`) e o desequilíbrio trata-se com pesos de classe. Cada modelo tem o seu `Pipeline`, ajustado só no treino.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# colunas enviesadas escolhidas só com o treino
sk = X_fit.skew()
skewed = [c for c in X_fit.columns
          if abs(sk[c]) > 1 and X_fit[c].min() >= 0 and X_fit[c].nunique() > 2]

def prep():
    return [
        ("log", ColumnTransformer([("log1p", FunctionTransformer(np.log1p), skewed)],
                                  remainder="passthrough")),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]

# XGBoost precisa de classes numéricas
le = LabelEncoder().fit(y_train)
y_fit_enc, y_val_enc = le.transform(y_fit), le.transform(y_val)

modelos = {
    "Logistic Regression": Pipeline(prep() + [("model", LogisticRegression(
        max_iter=300, class_weight="balanced", random_state=SEED))]),
    "Decision Tree": Pipeline(prep() + [("model", DecisionTreeClassifier(
        max_depth=15, class_weight="balanced", random_state=SEED))]),
    "Random Forest": Pipeline(prep() + [("model", RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1))]),
    "XGBoost": Pipeline(prep() + [("model", XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist",
        random_state=SEED, n_jobs=-1))]),
}

## 4. Treino e comparação na validação

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

resultados, previsoes = [], {}

for nome, pipe in modelos.items():
    t0 = time.time()
    if nome == "XGBoost":
        # desequilíbrio: pesos por amostra (equivalente ao class_weight="balanced")
        pipe.fit(X_fit, y_fit_enc, model__sample_weight=compute_sample_weight("balanced", y_fit_enc))
        pred = le.inverse_transform(pipe.predict(X_val))
    else:
        pipe.fit(X_fit, y_fit)
        pred = pipe.predict(X_val)
    previsoes[nome] = pred
    resultados.append({
        "modelo": nome,
        "F1 macro": round(f1_score(y_val, pred, average="macro"), 4),
        "accuracy": round(accuracy_score(y_val, pred), 4),
        "tempo (s)": round(time.time() - t0),
    })
    print(f"{nome}: F1 macro = {resultados[-1]['F1 macro']}  ({resultados[-1]['tempo (s)']}s)")

tabela = pd.DataFrame(resultados).sort_values("F1 macro", ascending=False).reset_index(drop=True)
tabela

## 5. Detalhe por classe do melhor modelo

In [ ]:
melhor = tabela.loc[0, "modelo"]
print("Melhor modelo:", melhor)
print(classification_report(y_val, previsoes[melhor], digits=3, zero_division=0))

## 6. Conclusões para o relatório

*(Preenche depois de correres o notebook.)*

- **Melhor modelo na validação:** …  (F1 macro = …)
- **Classes difíceis:** … (ver o detalhe por classe)
- **Comparação:** modelos baseados em árvores vs. Logistic Regression …
- **Limitações:** treino numa amostra; `Other_rare` tem poucos exemplos na validação.
- **Próximo passo (Módulo 7):** matriz de confusão, AUC-ROC, *hyperparameter tuning* do melhor modelo e avaliação final **uma única vez** no teste.